# 08b · The 2022 rate shock (task 8)

From 16 March 2022 the Federal Reserve raised rates by 525 bp in 16 months. Two things changed for
earnings releases:

1. discount-rate news dominated price moves, so the cash-flow news in a release may explain *less*
   of the reaction;
2. the *content* of releases changed (pricing power, input costs, interest expense, guidance cuts),
   vocabulary a model trained on 2006–2019 has rarely seen.

A text signal that weakens in 2022–23 and recovers afterwards points to (1); one that stays weak points
to (2) or to a model relying on past patterns, and for an LLM possibly on memorised pre-cutoff outcomes
(part 9).

| Regime | Dates (FOMC) |
|---|---|
| `pre_shock` | 2021-01-01 to 2022-03-15 |
| `hiking` | 2022-03-16 (first hike) to 2023-07-26 (last hike) |
| `post_hiking` | 2023-07-27 to 2025-12-31 |

**All three regimes are in the holdout.** This notebook does nothing until the holdout is frozen and
unlocked (`UNLOCK_HOLDOUT=1`). That is deliberate: looking at 2021–2025 returns before the freeze would
contaminate the test set.

In [1]:
# --- Setup (same in every notebook) ----------------------------------------
import os, sys, warnings
from pathlib import Path

try:                                   # on Colab: mount Drive and move to this folder
    from google.colab import drive
    drive.mount("/content/drive")
    os.chdir("/content/drive/MyDrive/S08 NLP/7_8_baselines_evaluation")
except ImportError:                    # on a laptop: run Jupyter from this folder
    pass

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "code"))

# SYNTHETIC = True runs everything on fake data with a planted effect and writes to
# results/_synthetic/ (never over the real results). Set STEP78_SYNTHETIC=1 or edit here.
SYNTHETIC = os.environ.get("STEP78_SYNTHETIC", "0") == "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

import settings as S
if SYNTHETIC:
    S.redirect_outputs("synthetic")
S.ensure_dirs()
import io_inputs as IO, outputs as OUT, pipeline as P
print("TEAM_DIR :", S.TEAM_DIR)
print("outputs  :", S.RESULTS)
print("mode     :", "SYNTHETIC DEMO" if SYNTHETIC else "real data")

TEAM_DIR : /Users/maximelacroix/Downloads/S08 NLP
outputs  : /Users/maximelacroix/Downloads/S08 NLP/7_8_baselines_evaluation/results
mode     : real data


In [2]:
UNLOCKED = os.environ.get("UNLOCK_HOLDOUT") == "1"
ev, texts, sectors, scores, prov = P.load_all(SYNTHETIC)
if UNLOCKED:
    panel = P.build_inputs(ev, texts, sectors, include_holdout=True)
else:
    print("Holdout locked: the rate-shock analysis will run after the freeze.")
    panel = P.build_inputs(ev, texts, sectors)

texts: 111 cleaned documents from 1 year files in /Users/maximelacroix/Downloads/S08 NLP/5_data cleaning/data/clean
texts: 98 more from the fallback export release_texts.csv (not cleaned by part 5: use for development only)


sectors: 100.0% of events have a point-in-time SIC
model scores: the cache is empty (part 6 has not run)
Holdout locked: the rate-shock analysis will run after the freeze.
panel: 217,611 events -> 69,028 in sample 'in_primary' without 2020
holdout guard: dropped 22,145 rows from 2021-2025


In [3]:
from rate_shock import rate_shock_analysis
from panel import cs_rank
sig = P.load_signals_aligned(panel, scores)
TEXTS = P.available_text_signals(sig)
d = panel.copy()
for t in TEXTS:
    d[t + "_rk"] = cs_rank(sig[t], d["season"])
by, inter = [], []
for t in TEXTS or [None]:
    if t is None:
        print("no text signal yet"); break
    r = rate_shock_analysis(d, t + "_rk", S.PRIMARY_OUTCOME)
    by.append(r["by_regime"]); inter.append(r["interaction"])
    display(r["by_regime"]); display(r["interaction"])
if UNLOCKED and by:
    OUT.save_table(pd.concat(by), "t8_rate_shock_by_regime", "Text signal by rate regime")
    OUT.save_table(pd.concat(inter), "t8_rate_shock_interactions", "Text x regime interactions")

no text signal yet
